# QA Agent Web UI (Notebook)

This notebook starts a Gradio Web UI on top of the existing CLI codebase.

Run order:
1. Run the initialization code cell below
2. Run the last cell to launch the UI
3. Open the local link shown in output (usually `http://127.0.0.1:7860`)

In [4]:
from pathlib import Path
import sys

# Ensure the notebook can import modules from the project root
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

try:
    import gradio as gr
except ImportError:
    %pip install -q gradio
    import gradio as gr

from main import initialize_agent

agent = initialize_agent()
print("QA Agent initialized for Web UI")

[2026-03-04 20:22:06,749] [main] [INFO] Initializing QA Agent components...
[2026-03-04 20:22:06,751] [src.retrievers.hybrid_retriever] [WARNING] acts_chunked_dir not found: data/acts_chunked
[2026-03-04 20:22:06,752] [src.retrievers.hybrid_retriever] [INFO] HybridRetriever initialized with 0 documents
[2026-03-04 20:22:06,810] [main] [INFO] QA Agent initialized successfully


QA Agent initialized for Web UI


In [ ]:
def _format_details(response):
    sources = "\n".join(f"- {src}" for src in sorted(set(response.sources))) if response.sources else "- (none)"
    citations = "\n".join(f"- {c}" for c in sorted(set(response.legal_citations))) if response.legal_citations else "- (none)"
    return (
        f"**Confidence**: {response.confidence_score:.1%}\n\n"
        f"**Processing time**: {response.processing_time:.2f}s\n\n"
        f"**Sources**\n{sources}\n\n"
        f"**Legal citations**\n{citations}"
    )


def ask_agent(user_query, history):
    query = (user_query or "").strip()
    if not query:
        return "", history, "Please enter a question."

    history = history or []

    try:
        response = agent.process_query(query)
        details = _format_details(response)
        history = history + [
            {"role": "user", "content": query},
            {"role": "assistant", "content": response.answer}
        ]
        return "", history, details
    except Exception as exc:
        error_message = str(exc)
        history = history + [
            {"role": "user", "content": query},
            {"role": "assistant", "content": "I encountered a temporary processing error. Please try again in a few seconds."}
        ]
        details = (
            "**Error**: Request failed.\n\n"
            f"**Technical detail**: {error_message}\n\n"
            "If this is a network SSL/proxy issue, retry the same query or verify your network/proxy settings."
        )
        return "", history, details


def clear_chat():
    agent.clear_history()
    return [], "Conversation history has been cleared."


with gr.Blocks(title="Singapore Legal & Tax QA Agent") as demo:
    gr.Markdown("## Singapore Legal & Tax QA Agent")
    gr.Markdown("Notebook-based Web UI built on top of the existing CLI pipeline")

    chatbot = gr.Chatbot(height=450, label="Conversation")
    details = gr.Markdown("Query metadata will be shown here.")

    with gr.Row():
        query_input = gr.Textbox(label="Your Question", placeholder="Ask about Singapore legal/tax rules...", scale=5)
        send_btn = gr.Button("Send", variant="primary", scale=1)

    clear_btn = gr.Button("Clear History")

    send_btn.click(ask_agent, inputs=[query_input, chatbot], outputs=[query_input, chatbot, details])
    query_input.submit(ask_agent, inputs=[query_input, chatbot], outputs=[query_input, chatbot, details])
    clear_btn.click(clear_chat, outputs=[chatbot, details])

demo.launch(inline=True, share=False)

[2026-03-04 20:22:09,626] [httpcore.connection] [DEBUG] connect_tcp.started host='127.0.0.1' port=7890 local_address=None timeout=3 socket_options=None
[2026-03-04 20:22:09,666] [httpcore.connection] [DEBUG] connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x168c8c530>
[2026-03-04 20:22:09,680] [httpcore.http11] [DEBUG] send_request_headers.started request=<Request [b'CONNECT']>
[2026-03-04 20:22:09,680] [httpcore.http11] [DEBUG] send_request_headers.complete
[2026-03-04 20:22:09,682] [httpcore.http11] [DEBUG] send_request_body.started request=<Request [b'CONNECT']>
[2026-03-04 20:22:09,682] [httpcore.http11] [DEBUG] send_request_body.complete
[2026-03-04 20:22:09,682] [asyncio] [DEBUG] Using selector: KqueueSelector
[2026-03-04 20:22:09,683] [httpcore.http11] [DEBUG] receive_response_headers.started request=<Request [b'CONNECT']>
[2026-03-04 20:22:09,684] [httpcore.http11] [DEBUG] receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'Connec

* Running on local URL:  http://127.0.0.1:7862


[2026-03-04 20:22:10,215] [httpcore.http11] [DEBUG] receive_response_headers.complete return_value=(b'HTTP/1.1', 502, b'Bad Gateway', [(b'Connection', b'close'), (b'Content-Length', b'0')])
[2026-03-04 20:22:10,216] [httpx] [INFO] HTTP Request: GET http://127.0.0.1:7862/gradio_api/startup-events "HTTP/1.1 502 Bad Gateway"
[2026-03-04 20:22:10,216] [httpcore.http11] [DEBUG] receive_response_body.started request=<Request [b'GET']>
[2026-03-04 20:22:10,216] [httpcore.http11] [DEBUG] receive_response_body.complete
[2026-03-04 20:22:10,217] [httpcore.http11] [DEBUG] response_closed.started
[2026-03-04 20:22:10,218] [httpcore.http11] [DEBUG] response_closed.complete


Exception: Couldn't start the app because 'http://127.0.0.1:7862/gradio_api/startup-events' failed (code 502). Check your network or proxy settings to ensure localhost is accessible.

[2026-03-04 20:22:10,866] [httpcore.proxy] [DEBUG] start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x17a711010>
[2026-03-04 20:22:10,868] [httpcore.http11] [DEBUG] send_request_headers.started request=<Request [b'GET']>
[2026-03-04 20:22:10,869] [httpcore.http11] [DEBUG] send_request_headers.complete
[2026-03-04 20:22:10,872] [httpcore.http11] [DEBUG] send_request_body.started request=<Request [b'GET']>
[2026-03-04 20:22:10,873] [httpcore.http11] [DEBUG] send_request_body.complete
[2026-03-04 20:22:10,874] [httpcore.http11] [DEBUG] receive_response_headers.started request=<Request [b'GET']>
[2026-03-04 20:22:11,218] [httpcore.http11] [DEBUG] receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 04 Mar 2026 12:22:11 GMT'), (b'Content-Type', b'application/json'), (b'Content-Length', b'21'), (b'Connection', b'keep-alive'), (b'Server', b'nginx/1.18.0'), (b'Access-Control-Allow-Origin', b'*')])
[2026-03-04 20:22:11,220] [htt